In [5]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3
from botocore.exceptions import NoCredentialsError
from dotenv import load_dotenv

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Obter as credenciais do MinIO do arquivo .env
minio_url = os.getenv('MINIO_ENDPOINT')
minio_access_key = os.getenv('MINIO_ACCESS_KEY')
minio_secret_key = os.getenv('MINIO_SECRET_KEY')
bucket_name = "bronze"

# Criar cliente MinIO
s3_client = boto3.client('s3',
                         endpoint_url=f'http://{minio_url}',
                         aws_access_key_id=minio_access_key,
                         aws_secret_access_key=minio_secret_key,
                         config=boto3.session.Config(signature_version='s3v4'))

# Baixar o arquivo CSV do bucket raw no MinIO
csv_path = "nafld1.csv"
local_csv_path = "nafld1_downloaded.csv"
try:
    s3_client.download_file("raw", csv_path, local_csv_path)
    print(f"Arquivo {csv_path} baixado com sucesso do bucket 'raw'.")
except Exception as e:
    print(f"Erro ao baixar o arquivo CSV: {e}")

# Carregar o arquivo CSV
df = pd.read_csv(local_csv_path)

# Selecionar colunas úteis para análise
cols_to_analyze = ['age', 'weight', 'height', 'bmi', 'futime', 'male', 'status']

# Verificar se todas as colunas estão presentes no dataframe
missing_cols = [col for col in cols_to_analyze if col not in df.columns]
if missing_cols:
    print(f"As seguintes colunas estão faltando no dataframe: {missing_cols}")
else:
    # Filtrar os dados para as colunas úteis
    df_filtered = df[cols_to_analyze]

    # Salvar o dataframe como arquivo Parquet
    parquet_path = "nafld1.parquet"
    df_filtered.to_parquet(parquet_path)

    # Carregar o arquivo Parquet para o bucket bronze no MinIO
    try:
        s3_client.upload_file(parquet_path, bucket_name, "nafld1.parquet")
        print(f"Arquivo {parquet_path} carregado com sucesso no bucket {bucket_name}.")
    except Exception as e:
        print(f"Erro ao carregar o arquivo Parquet para o MinIO: {e}")

Arquivo nafld1.csv baixado com sucesso do bucket 'raw'.
Arquivo nafld1.parquet carregado com sucesso no bucket bronze.
